# Session 4 — Bandits: an RL-based agent

**Salih Tutun, PhD** · DAT 5668 · Module 2 · Practice session (about 80 minutes)

I am Salih Tutun. We leave documents. Until I say **open the notebook**, this file stays closed. **Do not scroll.** Pairs talk first. Then type. After we run: **LLM**, **RL**, or **hybrid** — and one line for your project.

**Family this session: RL-based agent.** Repeated numeric choices and a reward. Session 3's LLM-based agent does not fit this job, and this bandit does not fit Session 3's documents.

![A bandit picks one FX sleeve](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-bandit.png)

![The agent chooses a; the environment returns r and s'](https://raw.githubusercontent.com/salihtutun/Agentic-AI_Autonomous_Decision_Making/main/assets/desk-agent-env.png)

## Real business problem — overnight FX sleeves

A **corporate treasury desk** parks overnight USD cash in **one** currency sleeve each business day: EUR, GBP, or JPY. Only the sleeve you chose pays that night's return. You never see the return of the sleeves you skipped. That is a multi-armed bandit.

The quotes are **real**: daily USD rates from [Frankfurter](https://www.frankfurter.app/) (public-apis, No Auth), 2024 Q1 snapshot in `fx-usd-sleeves.csv`. No employer data. No API key required.

| Word | Tonight |
| --- | --- |
| **AGENT** | ε-greedy (the **bandit**) chooses *a* |
| **ENVIRONMENT** | Frankfurter: only the sleeve you picked pays |
| State *s* | This visit: park cash in one sleeve |
| Observe | The return of the sleeve you picked — the others stay hidden |
| Decide | ε-greedy. RAG does not belong here. |
| Action *a* | EUR, GBP, or JPY |
| Reward *r* | That night's return (can be negative) |
| Next *s′* | Tomorrow's visit |

## Run of show (pair work)
- **10 min, slides** — Recap Session 3. Why this is not a document problem.
- **3 min PAIR** — Name the loop for this desk.
- **5 min** — Load real rates. Guess day 0 with your pair.
- **20 min PAIR** — Write ε-greedy and the sample-mean update until PASS.
- **12 min PAIR** — Compare ε = 0, 0.10, 0.40. Vote on a class epsilon.
- **15 min** — Debrief + framework. When a bandit is the wrong tool.

## The rule, before you code it

**ε-greedy.** Draw a number uniformly between 0 and 1. If it is below `epsilon`, return a random sleeve index (**explore**). Otherwise return the smallest index whose value is largest (**exploit**). `epsilon = 0` never explores. `epsilon = 1` never uses what it learned. `0.10` is the useful middle.

**Sample mean.** After reward `r` on sleeve index `a`:

- `counts[a] = counts[a] + 1`
- `values[a] = values[a] + (r - values[a]) / counts[a]`

**Regret** is the extra return the best mean sleeve would have earned, minus what you got. Code can be correct and regret still high — that is a finding.


## How we work in this notebook

I am Salih. I talk. You guess to a neighbor. Two voices share. Then we type **this cell only**. Do not scroll.

| You will see | What you do in class |
| --- | --- |
| **STOP** | Wait. Do not run the next cell until I say. |
| **PAIR (N min)** | Groups of 2 (or 3). One laptop. Talk the full time. Do not type until I say. |
| **This cell — ...** | I explain. Then we run it together. |
| **YOU FILL THIS** | Guess to a neighbor. Two people share. Then type `YOU TYPE HERE`. |
| **TODO** / `YOU WRITE THIS` | You write the function **with your pair**. Search for `TODO`. We stay until every line says PASS. |
| **RUN WITH ME** | Already written. Run it. Do not type in that cell. |
| **Family check** | Say **LLM**, **RL**, or **hybrid** — and one line for your project framework. |


## Stop 1 — PAIR (3 min). Name the loop before any code

**STOP. Do not scroll.** Groups of 2 (or 3). One laptop. Talk the full 3 minutes. Then type.

Agree on four short phrases for **tonight's desk** (not Session 3's CFPB ticket):


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Pair first (3 min). Then type. Short phrases.
state_tonight = ""       # YOU TYPE HERE — what the desk sees before it picks
action_set = ""          # YOU TYPE HERE — the three sleeves
reward_tonight = ""      # YOU TYPE HERE — what comes back after the close
next_state = ""          # YOU TYPE HERE — what happens tomorrow
print("state:", state_tonight or "(talk, then type)")
print("actions:", action_set or "(talk, then type)")
print("reward:", reward_tonight or "(talk, then type)")
print("next:", next_state or "(talk, then type)")


## Stop 2 — RUN WITH ME. Real Frankfurter rates

I run this. You watch. The agent does **not** use the mean table inside the loop. We print means only for evaluation (regret vs an oracle).


In [ ]:
# RUN WITH ME. Real Frankfurter USD rates → daily returns. No API key.
from pathlib import Path
import csv
import numpy as np


def course_data(name):
    for base in (Path("data"), Path("../../assets/data"), Path("assets/data"), Path("/content/data")):
        path = base / name
        if path.exists():
            return path
    raise FileNotFoundError(name)


def load_fx_levels_and_returns():
    """Load real Frankfurter USD quotes, then overnight returns per sleeve."""
    rows = list(csv.DictReader(course_data("fx-usd-sleeves.csv").open(encoding="utf-8")))
    dates = [row["date"] for row in rows]
    levels = {name: [float(row[name]) for row in rows] for name in ("EUR", "GBP", "JPY")}
    returns = {
        name: [(series[i] - series[i - 1]) / series[i - 1] for i in range(1, len(series))]
        for name, series in levels.items()
    }
    return dates, levels, returns


DATES, LEVELS, RETURNS = load_fx_levels_and_returns()
OFFERS = ["EUR", "GBP", "JPY"]
N_DAYS = len(RETURNS["EUR"])
# Oracle for evaluation only — the bandit never reads this inside decide.
TRUE_MEAN = {name: float(np.mean(RETURNS[name])) for name in OFFERS}


def sleeve_reward(offer, day):
    """One business day. Only the chosen sleeve's real return is observed."""
    return RETURNS[offer][day % N_DAYS]


def best_possible_rate():
    return max(TRUE_MEAN.values())


print("Source: Frankfurter daily USD quotes → overnight returns")
print("Calendar:", DATES[0], "→", DATES[-1], "| return days:", N_DAYS)
print("Sleeves:", OFFERS)
print()
print("First 5 overnight returns (talk about these next):")
print(f"{'day':>3} {'date':>12} {'EUR':>10} {'GBP':>10} {'JPY':>10}")
for day in range(5):
    print(
        f"{day:3d} {DATES[day + 1]:>12} "
        f"{RETURNS['EUR'][day]:10.5f} {RETURNS['GBP'][day]:10.5f} {RETURNS['JPY'][day]:10.5f}"
    )
print()
print("Oracle mean daily return (evaluation only):", {k: round(v, 5) for k, v in TRUE_MEAN.items()})
print("Oracle best mean sleeve:", max(TRUE_MEAN, key=TRUE_MEAN.get))


## Stop 3 — PAIR (3 min). Day 0 only

**STOP.** Look at day 0 in the printout. Talk with your pair:

1. Which sleeve had the best return on day 0?
2. If the desk is greedy forever after day 0, what goes wrong?

Then type. Do not open the TODO yet.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Pair first (3 min). Allowed sleeves: EUR, GBP, JPY.
day0_best_guess = ""  # YOU TYPE HERE — which sleeve won day 0?
greedy_risk = ""      # YOU TYPE HERE — one sentence: what goes wrong if we never explore?
print("Day 0 winner (your read):", day0_best_guess or "(look at the table, then type)")
print("Greedy risk:", greedy_risk or "(one sentence)")


## Stop 4 — PAIR (20 min). Write the bandit until PASS

Search for `TODO`. One laptop. Talk, then type together.

1. Complete `epsilon_greedy`.
2. Complete the sample-mean update inside `run_bandit`.
3. Run the **check** cell. Stay until every line says **PASS**.


In [ ]:
# TODO — you write this with your pair. Search for TODO.
def epsilon_greedy(values, epsilon, rng):
    """Return an action index. Explore with probability epsilon; else exploit."""
    # TODO: if rng.random() < epsilon, return a random index in 0 .. len(values)-1
    # TODO: otherwise return the smallest index of the largest value (np.argmax)
    return 0


def run_bandit(epsilon, n_visits=None, seed=7):
    """Park overnight cash each day. Only the chosen sleeve's return is observed."""
    n_visits = n_visits or N_DAYS
    rng = np.random.default_rng(seed)  # fresh seed each run so epsilon comparisons are fair
    values = np.zeros(len(OFFERS))
    counts = np.zeros(len(OFFERS))
    rewards = []
    for day in range(n_visits):
        index = epsilon_greedy(values, epsilon, rng)
        offer = OFFERS[index]
        reward = sleeve_reward(offer, day)
        # TODO: update counts[index] and values[index] with the sample-mean formula
        # counts[index] = ...
        # values[index] = ...
        rewards.append(reward)
    return values, counts, np.cumsum(rewards)


values, counts, cumulative = run_bandit(0.10)
print("values", np.round(values, 5))
print("counts", counts.astype(int))
print("cumulative return", round(float(cumulative[-1]), 5) if len(cumulative) else 0)
print("If counts are all zero, the update TODO is still empty — keep going.")


**This cell — the checks.** Run with me after your pair finishes the TODOs. Do not edit the expected behavior.


In [ ]:
# RUN WITH ME — do not edit. Re-run until every line says PASS.
def check_bandit():
    rng = np.random.default_rng(0)
    # Exploit only: must pick the unique max, then the smallest index on a tie.
    if epsilon_greedy(np.array([0.1, 0.5, 0.2]), 0.0, rng) != 1:
        print("NOT YET epsilon_greedy — with epsilon=0 pick the largest value")
        return
    if epsilon_greedy(np.array([0.5, 0.5, 0.1]), 0.0, rng) != 0:
        print("NOT YET epsilon_greedy — ties break toward the smallest index (np.argmax)")
        return
    # Explore only: every index should appear if we draw enough times.
    picks = {epsilon_greedy(np.array([0.9, 0.1, 0.0]), 1.0, rng) for _ in range(80)}
    if picks != {0, 1, 2}:
        print("NOT YET epsilon_greedy — with epsilon=1 return a random index each time")
        return
    values, counts, cumulative = run_bandit(0.0, seed=7)
    if counts.sum() == 0:
        print("NOT YET run_bandit — write the sample-mean update (counts and values)")
        return
    if int(counts[0]) != N_DAYS or int(counts[1]) != 0 or int(counts[2]) != 0:
        print("NOT YET run_bandit — epsilon=0 with seed 7 should lock on EUR (index 0) all days")
        return
    expected = float(np.mean(RETURNS["EUR"]))
    if abs(float(values[0]) - expected) > 1e-9:
        print("NOT YET run_bandit — values[0] should be the sample mean of EUR rewards")
        return
    values10, counts10, _ = run_bandit(0.10, seed=7)
    if counts10.min() < 1:
        print("NOT YET — epsilon=0.10 should still sample every sleeve at least once with seed 7")
        return
    print("PASS epsilon_greedy")
    print("PASS sample-mean update")
    print("PASS epsilon=0 locks on EUR; epsilon=0.10 still explores")
    print("counts at 0.10:", counts10.astype(int).tolist(), "| learned:", OFFERS[int(np.argmax(values10))])


check_bandit()


## Stop 5 — PAIR (5 min). Compare three epsilons

Run the report. Then talk: what changed from 0 → 0.10 → 0.40? Type one sentence.


In [ ]:
# RUN WITH ME after PASS. Compare explore / exploit on real 2024 Q1 returns.
def regret_so_far(cumulative):
    """Extra return the best mean sleeve would have collected."""
    visits = np.arange(1, len(cumulative) + 1)
    best = best_possible_rate() * visits
    return best - cumulative


def report(epsilon, seed=7):
    values, counts, cumulative = run_bandit(epsilon, seed=seed)
    if counts.sum() == 0:
        print(f"epsilon={epsilon}: NOT YET — the loop never updated the counts")
        return
    gap = float(regret_so_far(cumulative)[-1])
    learned = OFFERS[int(np.argmax(values))]
    print(
        f"epsilon={epsilon}: learned={learned}  counts={counts.astype(int).tolist()}  "
        f"cumulative={float(cumulative[-1]):.4f}  regret vs oracle {gap:.4f}"
    )


print("Oracle best mean sleeve:", max(TRUE_MEAN, key=TRUE_MEAN.get), TRUE_MEAN)
for epsilon in (0.0, 0.10, 0.40):
    report(epsilon)


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Pair first. One sentence on what changed across 0 / 0.10 / 0.40.
what_changed = ""  # YOU TYPE HERE — at least 40 characters
if len(what_changed) < 40:
    print("NOT YET — talk with your pair, then type one sentence (40+ characters)")
else:
    print("Your read:", what_changed)


## Stop 6 — PAIR then class vote. Pick an epsilon

**STOP.** Neighbor 20s. Two voices. 0 never explores. 1 never settles. 0.10 is the useful middle. Agree on a number, type it, run.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Neighbor first. A number from 0 to 1.
class_epsilon = None  # YOU TYPE HERE, for example 0.10
if class_epsilon is None:
    print("Agree on an epsilon, type it, and run again.")
else:
    report(class_epsilon)
    print("Say whether this was too shy or too restless.")


## Stop 7 — PAIR (4 min). Debrief

Talk first. Then type short answers.

1. What goes wrong at epsilon 0 if the first unlucky days make GBP look bad?
2. Why is a CFPB billing dispute a poor job for this bandit?
3. Name a business test you would **not** run as a bandit (one bad pull is too costly).


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Pair first. Short answers.
epsilon0_fails_because = ""  # YOU TYPE HERE
cfpb_is_wrong_job_because = ""  # YOU TYPE HERE
would_not_bandit = ""  # YOU TYPE HERE — e.g. large wire / medical denial / auto-refund
print("1.", epsilon0_fails_because or "(talk, then type)")
print("2.", cfpb_is_wrong_job_because or "(talk, then type)")
print("3.", would_not_bandit or "(talk, then type)")


## Add this to your framework — STOP. Do not scroll.

Guess to a neighbor (20 seconds): after today, is **your** project **LLM**, **RL**, or **hybrid**? Two voices share. Then type one sentence you will keep.

Hint: an overnight FX sleeve is **RL**. Session 3's CFPB complaint is **LLM**. Hybrid later: learned sleeve on the usual day; a language agent on the exception email.


In [ ]:
# FILL WITH THE CLASS. >>> YOU FILL THIS <<<
# Neighbor first. Two voices. Then type.
framework_family = ""  # YOU TYPE HERE: LLM, RL, or hybrid
framework_line = ""    # YOU TYPE HERE: one sentence you will keep
print("Family:", framework_family or "(say it, then type it)")
print("I will keep:", framework_line or "(one sentence)")
